# 08 — Automated Commentary & Report

Turns the metrics into research-note prose, e.g. *"Berkshire increased exposure to Chevron while trimming Occidental. Apple remained the largest holding, representing 24.8% of assets. Portfolio turnover declined compared with the previous quarter."*

Every sentence is generated from the data — swap the CIK in `config.py` and the same code narrates any manager. The full markdown report is written to `reports/`.

In [1]:
# --- setup (run this cell first, every session) ---
import os, sys, pathlib

# Option A: Google Drive (data persists across sessions -- recommended)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: uploaded zip directly to Colab (data lost on disconnect)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# Option C: running locally
PROJECT_ROOT = pathlib.Path(__file__).resolve().parents[1] if "__file__" in dir() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# Install dependencies if needed (uncomment on Colab)
# import subprocess; subprocess.run(["pip", "install", "-q", "pyarrow", "requests", "pandas", "matplotlib", "numpy"])

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", PROJECT_ROOT)
print("Ready.")

Project root: /content/13F-Analytics
Ready.


In [2]:
from src import commentary, config
from src.utils import load_parquet

holdings = load_parquet(config.HOLDINGS_PARQUET)
tx = load_parquet(config.TRANSACTIONS_PARQUET)
summary = load_parquet(config.PORTFOLIO_PARQUET)

latest = summary["quarter"].iloc[-1]
print(commentary.quarterly_commentary(latest, holdings, tx, summary))

Berkshire Hathaway reported $263.1B of U.S.-listed holdings for 2026Q1, down 4.0% from the prior quarter across 29 positions. APPLE INC remained the largest holding, representing 22.0% of reported assets; the top five positions accounted for 67.1%. The firm added to ALPHABET INC ($10.0B). It trimmed BANK AMERICA CORP ($-3.4B). Portfolio turnover increased compared with the previous quarter (5.5% vs 1.8%).


In [3]:
report_md = commentary.full_report(holdings, tx, summary)
out = config.PROJECT_ROOT / "reports" / "portfolio_review.md"
out.write_text(report_md, encoding="utf-8")
print("wrote", out)
print()
print(report_md[:1500])

wrote /content/13F-Analytics/reports/portfolio_review.md

# Berkshire Hathaway — 13F Portfolio Review

## 2024Q3

Berkshire Hathaway reported $266.4B of U.S.-listed holdings for 2024Q3, down 4.9% from the prior quarter across 40 positions. APPLE INC remained the largest holding, representing 26.2% of reported assets; the top five positions accounted for 70.9%. The firm initiated a new position in SIRIUS XM HOLDINGS INC ($2.5B). It trimmed APPLE INC ($-14.3B).

## 2024Q4

Berkshire Hathaway reported $267.2B of U.S.-listed holdings for 2024Q4, up 0.3% from the prior quarter across 38 positions. APPLE INC remained the largest holding, representing 28.1% of reported assets; the top five positions accounted for 71.9%. The firm initiated a new position in CONSTELLATION BRANDS INC ($1.2B). It trimmed CITIGROUP INC ($-2.4B). Portfolio turnover declined compared with the previous quarter (0.8% vs 1.2%).

## 2025Q1

Berkshire Hathaway reported $1.1B of U.S.-listed holdings for 2025Q1, down 99.6%

### Optional next step — Parquet → Supabase
The `data/processed/*.parquet` files are the project's "database". To productionize, replace the parquet layer with Supabase/BigQuery tables: each notebook writes to a table instead of a file, everything else stays identical — that's the payoff of the *notebooks depend on data + functions, never on each other* rule.